In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 14.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import numpy as np
from datasets import load_dataset, concatenate_datasets, Dataset

# Load the CNN/Daily Mail dataset
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

# Function to summarize a single article using TF-IDF (extractive summarization)
def summarize_article(article):
    # Split the article into sentences
    sentences = article.split('. ')

    # Initialize TF-IDF Vectorizer
    vectorizer = TfidfVectorizer(stop_words='english')

    # Create the TF-IDF matrix
    tfidf_matrix = vectorizer.fit_transform(sentences)

    # Compute pairwise cosine similarity between sentences
    cosine_similarities = cosine_similarity(tfidf_matrix, tfidf_matrix)

    # Get the most important sentence by selecting the one with the highest average similarity
    sentence_scores = cosine_similarities.sum(axis=1)

    # Sort sentences by their scores
    ranked_sentences = [sentences[i] for i in np.argsort(sentence_scores, axis=0)[::-1]]

    # Take the top 3 sentences as the summary
    summary = '. '.join(ranked_sentences[:3])

    return summary

# Function to process and summarize the data
def process_and_summarize(data, split_name):
    original_articles = []
    summarized_articles = []

    # Process each article in the dataset
    for article in data:
        original_articles.append(article['article'])
        summary = summarize_article(article['article'])
        summarized_articles.append(summary)

    # Create a DataFrame with the original and summarized articles
    df = pd.DataFrame({
        'original_article': original_articles,
        'summarized_article': summarized_articles
    })

    # Save the results to a CSV file (different files for train and test)
    df.to_csv(f'summarized_{split_name}.csv', index=False)
    print(f"Summarization complete for {split_name}! The data has been saved to 'summarized_{split_name}.csv'.")

# Combine train and test datasets
all_articles = concatenate_datasets([dataset['train'], dataset['test']])

# Extract articles as a list of dictionaries
all_articles_list = [article for article in all_articles]

# Split data into training and test sets
train_data_list, test_data_list = train_test_split(all_articles_list, test_size=0.2, random_state=10)

# Convert lists back to Hugging Face datasets
train_data = Dataset.from_list(train_data_list)
test_data = Dataset.from_list(test_data_list)

# Summarize and save both training and test data
process_and_summarize(train_data, 'train')
process_and_summarize(test_data, 'test')


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Summarization complete for train! The data has been saved to 'summarized_train.csv'.
Summarization complete for test! The data has been saved to 'summarized_test.csv'.
